<a href="https://colab.research.google.com/github/poojabisht10/Test_Summarization_Topsis/blob/main/Text_Summarization.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import pandas as pd

data = {
    "Model": [
        "facebook/bart-large-cnn",
        "ARTeLab/it5-summarization-fanpage-64",
        "google/pegasus-xsum",
        "pszemraj/led-large-book-summary"
    ],
    "ROUGE-1": [42.949,33.5599,21.810, 31.731],
    "ROUGE-2": [20.815, 15.7432, 4.253, 5.331],
    "ROUGE-L": [30.619, 25.5253, 17.447, 16.146],
    "loss": [2.529, 1.4897, 3.032, 4.816],
    "gen_len": [78.587, 57.2958, 20.312, 154.904]
}

df = pd.DataFrame(data)

weights = np.array([0.25, 0.25, 0.20, 0.15, 0.15])
impact = np.array([1, 1, 1, -1, -1])

decision_matrix = df.iloc[:, 1:].values
norm_matrix = decision_matrix / np.sqrt((decision_matrix ** 2).sum(axis=0))

weighted_matrix = norm_matrix * weights

ideal_best = np.where(impact == 1, weighted_matrix.max(axis=0), weighted_matrix.min(axis=0))
ideal_worst = np.where(impact == 1, weighted_matrix.min(axis=0), weighted_matrix.max(axis=0))

distance_best = np.sqrt(((weighted_matrix - ideal_best) ** 2).sum(axis=1))
distance_worst = np.sqrt(((weighted_matrix - ideal_worst) ** 2).sum(axis=1))

topsis_score = distance_worst / (distance_best + distance_worst)

df["TOPSIS_Score"] = topsis_score
df["Rank"] = df["TOPSIS_Score"].rank(ascending=False)

print(df.sort_values("Rank"))

                                  Model  ROUGE-1  ROUGE-2  ROUGE-L    loss  \
0               facebook/bart-large-cnn  42.9490  20.8150  30.6190  2.5290   
1  ARTeLab/it5-summarization-fanpage-64  33.5599  15.7432  25.5253  1.4897   
2                   google/pegasus-xsum  21.8100   4.2530  17.4470  3.0320   
3       pszemraj/led-large-book-summary  31.7310   5.3310  16.1460  4.8160   

    gen_len  TOPSIS_Score  Rank  
0   78.5870      0.790296   1.0  
1   57.2958      0.703810   2.0  
2   20.3120      0.388022   3.0  
3  154.9040      0.154502   4.0  


In [ ]:
from tabulate import tabulate

df_sorted = df.sort_values("Rank")

print(tabulate(df_sorted, headers='keys', tablefmt='grid', showindex=False))

+--------------------------------------+-----------+-----------+-----------+--------+-----------+----------------+--------+
| Model                                |   ROUGE-1 |   ROUGE-2 |   ROUGE-L |   loss |   gen_len |   TOPSIS_Score |   Rank |
+======================================+===========+===========+===========+========+===========+================+========+
| facebook/bart-large-cnn              |   42.949  |   20.815  |   30.619  | 2.529  |   78.587  |       0.790296 |      1 |
+--------------------------------------+-----------+-----------+-----------+--------+-----------+----------------+--------+
| ARTeLab/it5-summarization-fanpage-64 |   33.5599 |   15.7432 |   25.5253 | 1.4897 |   57.2958 |       0.70381  |      2 |
+--------------------------------------+-----------+-----------+-----------+--------+-----------+----------------+--------+
| google/pegasus-xsum                  |   21.81   |    4.253  |   17.447  | 3.032  |   20.312  |       0.388022 |      3 |
+-------